# TensorFlow Implementation (EXTRA)

**Chapter 7: Predicting Soccer Outcomes with Deep Learning**

## Learning Objectives

- Understand TensorFlow/Keras framework
- Implement the same models from PyTorch in TensorFlow
- Compare PyTorch vs TensorFlow syntax and workflows
- Learn when to use which framework
- Understand the trade-offs between frameworks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, losses
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## PyTorch vs TensorFlow: Key Differences

### PyTorch
- **Dynamic computation graph**: More flexible, easier debugging
- **Pythonic**: Feels more like writing Python code
- **Research-friendly**: Popular in academia
- **Explicit**: You control every step

### TensorFlow/Keras
- **High-level API (Keras)**: Faster prototyping
- **Production-ready**: Better deployment tools
- **Industry standard**: Popular in production environments
- **Abstracted**: More automated, less boilerplate

### When to Use Which?
- **PyTorch**: Research, custom architectures, learning deep learning
- **TensorFlow**: Production deployment, mobile/edge devices, large-scale systems

Both are excellent choices! The best framework is the one you're most comfortable with.

## Generate Data (Same as PyTorch Notebooks)

In [ ]:
# Generate simulated match data
np.random.seed(42)
n_matches = 1000

# Features
shots_home = np.random.poisson(12, n_matches)
possession_home = np.random.normal(50, 15, n_matches)
xg_home = np.random.gamma(2, 0.6, n_matches)
corners_home = np.random.poisson(5, n_matches)
fouls_home = np.random.poisson(10, n_matches)

shots_away = np.random.poisson(10, n_matches)
possession_away = 100 - possession_home
xg_away = np.random.gamma(1.5, 0.6, n_matches)
corners_away = np.random.poisson(4, n_matches)
fouls_away = np.random.poisson(11, n_matches)

df = pd.DataFrame({
    'shots_home': shots_home,
    'possession_home': possession_home,
    'xg_home': xg_home,
    'corners_home': corners_home,
    'fouls_home': fouls_home,
    'shots_away': shots_away,
    'possession_away': possession_away,
    'xg_away': xg_away,
    'corners_away': corners_away,
    'fouls_away': fouls_away
})

# Generate outcomes
win_probability = 1 / (1 + np.exp(-(0.3 * (df['xg_home'] - df['xg_away']) + 
                                     0.02 * (df['shots_home'] - df['shots_away']) +
                                     0.01 * (df['possession_home'] - 50))))
df['home_win'] = (np.random.random(n_matches) < win_probability).astype(int)

def generate_ftr(row):
    score_diff = row['xg_home'] - row['xg_away'] + np.random.normal(0, 0.5)
    if score_diff > 0.5:
        return 'H'
    elif score_diff < -0.5:
        return 'A'
    else:
        return 'D'

df['FTR'] = df.apply(generate_ftr, axis=1)
df['FTHG'] = np.random.poisson(df['xg_home'])
df['FTAG'] = np.random.poisson(df['xg_away'])

print(f"Dataset shape: {df.shape}")
print(f"\nHome win rate: {df['home_win'].mean():.2%}")

---

# Task 1: Binary Classification (TensorFlow)

## Predicting Home Team Wins

In [ ]:
# Prepare data
feature_cols = ['shots_home', 'possession_home', 'xg_home', 'corners_home', 'fouls_home',
                'shots_away', 'possession_away', 'xg_away', 'corners_away', 'fouls_away']

X = df[feature_cols].values
y_binary = df['home_win'].values

# Standardize
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")

## Build Model with Keras Sequential API

Keras offers two ways to build models:
1. **Sequential API**: Simple, linear stack of layers (we'll use this)
2. **Functional API**: More flexible, for complex architectures

In [ ]:
# Build binary classification model
model_binary_tf = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu', name='hidden'),
    layers.Dense(1, activation='sigmoid', name='output')
], name='BinaryClassifier')

# Compile model
model_binary_tf.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss=losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

# Model summary
model_binary_tf.summary()

### PyTorch vs TensorFlow Comparison

**PyTorch:**
```python
class BinaryMLP(nn.Module):
    def __init__(self, input_size, hidden_size=16):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
```

**TensorFlow/Keras:**
```python
model = models.Sequential([
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
```

✅ **TensorFlow is more concise** - less boilerplate code

✅ **PyTorch is more explicit** - you see every operation

## Train the Model

In [ ]:
# Train model
history_binary = model_binary_tf.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0  # Set to 1 to see progress
)

print("Training complete!")
print(f"Final training accuracy: {history_binary.history['accuracy'][-1]:.3f}")
print(f"Final validation accuracy: {history_binary.history['val_accuracy'][-1]:.3f}")

In [ ]:
# Visualize training
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_binary.history['loss'], label='Training Loss')
axes[0].plot(history_binary.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Progress - Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history_binary.history['accuracy'], label='Training Accuracy')
axes[1].plot(history_binary.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training Progress - Accuracy', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluate the Model

In [ ]:
# Evaluate on test set
test_loss, test_acc = model_binary_tf.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc:.3f} ({test_acc*100:.1f}%)")

# Make predictions
y_pred_probs = model_binary_tf.predict(X_test, verbose=0)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Win', 'Win']))

---

# Task 2: Multi-Class Classification (TensorFlow)

## Predicting Win/Draw/Loss

In [ ]:
# Prepare multi-class data
le = LabelEncoder()
y_multiclass = le.fit_transform(df['FTR'])

X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X, y_multiclass, test_size=0.2, random_state=42
)

print(f"Classes: {le.classes_}")
print(f"Training set: {len(X_train_mc)} matches")

In [ ]:
# Build multi-class model
model_multiclass_tf = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax')  # 3 classes with softmax
], name='MultiClassifier')

# Compile
model_multiclass_tf.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss=losses.SparseCategoricalCrossentropy(),  # for integer labels
    metrics=['accuracy']
)

model_multiclass_tf.summary()

In [ ]:
# Train
history_mc = model_multiclass_tf.fit(
    X_train_mc, y_train_mc,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("Training complete!")
print(f"Final validation accuracy: {history_mc.history['val_accuracy'][-1]:.3f}")

In [ ]:
# Evaluate
test_loss_mc, test_acc_mc = model_multiclass_tf.evaluate(X_test_mc, y_test_mc, verbose=0)
print(f"Test Accuracy: {test_acc_mc:.3f} ({test_acc_mc*100:.1f}%)")

# Predictions
y_pred_mc = model_multiclass_tf.predict(X_test_mc, verbose=0)
y_pred_classes = np.argmax(y_pred_mc, axis=1)

print("\nClassification Report:")
print(classification_report(y_test_mc, y_pred_classes, target_names=le.classes_))

---

# Task 3: Poisson Regression (TensorFlow)

## Predicting Home Goals

In [ ]:
# Prepare goal data
y_goals = df['FTHG'].values

X_train_goals, X_test_goals, y_train_goals, y_test_goals = train_test_split(
    X, y_goals, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train_goals)} matches")
print(f"Average goals: {y_train_goals.mean():.2f}")

In [ ]:
# Build Poisson regression model
model_poisson_tf = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='softplus')  # ensures positive output
], name='PoissonRegression')

# Custom Poisson loss function
def poisson_loss(y_true, y_pred):
    return tf.reduce_mean(y_pred - y_true * tf.math.log(y_pred + 1e-8))

# Compile
model_poisson_tf.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss=poisson_loss,
    metrics=['mae']
)

model_poisson_tf.summary()

In [ ]:
# Train with early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history_poisson = model_poisson_tf.fit(
    X_train_goals, y_train_goals,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

print(f"Training stopped at epoch {len(history_poisson.history['loss'])}")
print(f"Best validation MAE: {min(history_poisson.history['val_mae']):.4f}")

In [ ]:
# Evaluate
y_pred_goals = model_poisson_tf.predict(X_test_goals, verbose=0).flatten()

mae = mean_absolute_error(y_test_goals, y_pred_goals)
rmse = np.sqrt(np.mean((y_test_goals - y_pred_goals)**2))

print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"\nOn average, predictions are off by {mae:.2f} goals")

---

# Task 4: Multi-Task Learning (TensorFlow)

## Predicting Both Home and Away Goals

For multi-output models, we'll use the **Functional API** instead of Sequential.

In [ ]:
# Prepare data for both outputs
y_home_goals = df['FTHG'].values
y_away_goals = df['FTAG'].values

X_train_mtl, X_test_mtl, y_home_train, y_home_test, y_away_train, y_away_test = train_test_split(
    X, y_home_goals, y_away_goals, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train_mtl)} matches")

In [ ]:
# Build multi-task model using Functional API
input_layer = layers.Input(shape=(X_train.shape[1],), name='input')

# Shared layers
shared = layers.Dense(64, activation='relu', name='shared_1')(input_layer)
shared = layers.Dense(32, activation='relu', name='shared_2')(shared)

# Home goals head
home_hidden = layers.Dense(32, activation='relu', name='home_hidden')(shared)
home_output = layers.Dense(1, activation='softplus', name='home_goals')(home_hidden)

# Away goals head
away_hidden = layers.Dense(32, activation='relu', name='away_hidden')(shared)
away_output = layers.Dense(1, activation='softplus', name='away_goals')(away_hidden)

# Create model with multiple outputs
model_mtl_tf = models.Model(
    inputs=input_layer,
    outputs=[home_output, away_output],
    name='MultiTaskModel'
)

model_mtl_tf.summary()

### Functional API Benefits

The Functional API allows:
- ✅ Multiple inputs and outputs
- ✅ Shared layers
- ✅ Complex architectures (residual connections, etc.)
- ✅ More flexibility than Sequential

In [ ]:
# Compile with separate losses for each output
model_mtl_tf.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss={
        'home_goals': poisson_loss,
        'away_goals': poisson_loss
    },
    metrics={
        'home_goals': ['mae'],
        'away_goals': ['mae']
    }
)

print("Model compiled with separate losses for each task")

In [ ]:
# Train
history_mtl = model_mtl_tf.fit(
    X_train_mtl,
    {'home_goals': y_home_train, 'away_goals': y_away_train},
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("Training complete!")
print(f"Final home goals MAE: {history_mtl.history['home_goals_mae'][-1]:.4f}")
print(f"Final away goals MAE: {history_mtl.history['away_goals_mae'][-1]:.4f}")

In [ ]:
# Evaluate
home_pred, away_pred = model_mtl_tf.predict(X_test_mtl, verbose=0)

print("Multi-Task Model Performance:")
print("\nHome Goals:")
print(f"  MAE: {mean_absolute_error(y_home_test, home_pred):.4f}")
print(f"  RMSE: {np.sqrt(np.mean((y_home_test - home_pred.flatten())**2)):.4f}")

print("\nAway Goals:")
print(f"  MAE: {mean_absolute_error(y_away_test, away_pred):.4f}")
print(f"  RMSE: {np.sqrt(np.mean((y_away_test - away_pred.flatten())**2)):.4f}")

In [ ]:
# Sample predictions
print("Sample Match Predictions:")
print("\nActual Score | Predicted Score (λ)")
print("-" * 40)
for i in range(10):
    actual_home = int(y_home_test[i])
    actual_away = int(y_away_test[i])
    pred_home = home_pred[i][0]
    pred_away = away_pred[i][0]
    
    print(f"{actual_home:^5}-{actual_away:<5} | {pred_home:^7.2f}-{pred_away:<7.2f}")

## Summary: PyTorch vs TensorFlow

### Similarities
- Both can build the same models
- Both support GPU acceleration
- Both have large communities and ecosystems
- Both are production-ready

### Key Differences

| Aspect | PyTorch | TensorFlow/Keras |
|--------|---------|------------------|
| **API Style** | More explicit, Pythonic | More abstracted, concise |
| **Learning Curve** | Steeper initially | Easier to start |
| **Debugging** | Easier (dynamic graph) | Harder (static graph) |
| **Deployment** | Improving | Mature (TF Serving, TF Lite) |
| **Research** | Very popular | Growing |
| **Industry** | Growing | Very popular |
| **Mobile/Edge** | Limited | Excellent (TF Lite) |

### Our Recommendation

- **Learning deep learning?** Start with PyTorch for better understanding
- **Quick prototyping?** TensorFlow/Keras for less boilerplate
- **Production deployment?** TensorFlow for better tooling
- **Research?** PyTorch for flexibility
- **Both?** Learn both! They're more similar than different

### Code Comparison Summary

**Model Definition:**
- PyTorch: More verbose, explicit forward pass
- TensorFlow: More concise, automatic forward pass

**Training:**
- PyTorch: Manual loop, explicit backward pass
- TensorFlow: `.fit()` method handles everything

**Evaluation:**
- PyTorch: Manual loop with `torch.no_grad()`
- TensorFlow: `.evaluate()` and `.predict()` methods

## Practice Exercises

1. **Convert a PyTorch model**: Take one of your PyTorch models and convert it to TensorFlow.

2. **Callbacks**: Explore TensorFlow callbacks (ModelCheckpoint, ReduceLROnPlateau, TensorBoard).

3. **Custom layers**: Create a custom layer in TensorFlow and use it in a model.

4. **Model saving**: Save and load models in both frameworks. Compare file formats.

5. **Performance comparison**: Train the same model in both frameworks and compare training time.

6. **Functional API**: Build a complex architecture (e.g., with skip connections) using the Functional API.

7. **Mixed precision**: Implement mixed precision training in TensorFlow for faster training.